# Data Preprocessing & Feature Engineering
### Student Academic Risk Prediction — UCI "Predict Students' Dropout and Academic Success"

This notebook implements the preprocessing and feature-engineering strategy step by step:
load & profile → clean → engineer features → encode → scale → select features → assemble into a single reusable pipeline.

**Before running:** upload `data.csv` (the UCI dataset export, semicolon-delimited) using the cell below, or mount Google Drive and point `DATA_PATH` to it.

## 0. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Option A: upload directly in Colab
from google.colab import files
uploaded = files.upload()   # choose data.csv when prompted
DATA_PATH = list(uploaded.keys())[0] if uploaded else 'data.csv'

In [ ]:
# Option B (alternative): mount Google Drive instead of uploading each time
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/path/to/data.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

## 1. Load and Profile the Data

In [ ]:
df = pd.read_csv(DATA_PATH, sep=';')
df.columns = [c.strip() for c in df.columns]   # strip stray whitespace/tabs in headers

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
missing_rate = df.isna().mean().sort_values(ascending=False)
print('Total missing cells:', df.isna().sum().sum())
missing_rate.head(10)

In [ ]:
dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {dupes}')
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
target_dist = df['Target'].value_counts(normalize=True).round(3)
print(target_dist)
sns.countplot(x='Target', data=df, order=df['Target'].value_counts().index)
plt.title('Target class distribution')
plt.show()

In [ ]:
df.describe().T.round(2)

## 1.3 Descriptive Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Admission grade'], kde=True, ax=axes[0])
axes[0].set_title('Admission grade distribution')

sns.boxplot(x='Target', y='Curricular units 1st sem (grade)', data=df, ax=axes[1])
axes[1].set_title('1st-semester grade by outcome')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Correlation heatmap (numeric columns)')
plt.show()

## 2. Data Cleaning
Missing-value imputers are defined here even though this dataset happens to be complete —
this keeps the pipeline robust to a different pull of the same data.

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

### 2.3 Outlier Detection (IQR rule) — investigate, don't blindly drop

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lo) | (series > hi)).sum()

for col in ['Admission grade', 'Age at enrollment',
            'Curricular units 1st sem (grade)', 'Unemployment rate']:
    print(col, '->', iqr_outliers(df[col]), 'flagged of', len(df))

## 3. Feature Engineering

### 3.1 New Features

In [ ]:
eps = 1e-6
df['approval_rate_s1'] = df['Curricular units 1st sem (approved)'] / \
                          (df['Curricular units 1st sem (enrolled)'] + eps)
df['approval_rate_s2'] = df['Curricular units 2nd sem (approved)'] / \
                          (df['Curricular units 2nd sem (enrolled)'] + eps)

df['grade_trend'] = df['Curricular units 2nd sem (grade)'] - \
                     df['Curricular units 1st sem (grade)']

df['total_enrolled'] = df['Curricular units 1st sem (enrolled)'] + \
                        df['Curricular units 2nd sem (enrolled)']
df['total_approved'] = df['Curricular units 1st sem (approved)'] + \
                        df['Curricular units 2nd sem (approved)']

df['zero_approval_flag'] = ((df['Curricular units 1st sem (approved)'] == 0) |
                             (df['Curricular units 2nd sem (approved)'] == 0)).astype(int)

df['age_bucket'] = pd.cut(df['Age at enrollment'],
                           bins=[0, 20, 25, 35, 100],
                           labels=['18-20', '21-25', '26-35', '36+'])

df['parent_edu_max'] = df[["Mother's qualification", "Father's qualification"]].max(axis=1)

df[['approval_rate_s1', 'approval_rate_s2', 'grade_trend',
    'zero_approval_flag', 'age_bucket', 'parent_edu_max']].head()

### 3.2 Variable Transformation (skew correction, applied on the linear-model branch)

In [ ]:
skewed_cols = ['Curricular units 1st sem (credited)',
               'Curricular units 2nd sem (credited)']
df_log = df.copy()
for col in skewed_cols:
    df_log[col + '_log'] = np.log1p(df_log[col])

df_log[[c + '_log' for c in skewed_cols]].describe().round(2)

### 3.3 / 3.4 Define Column Groups for Encoding and Scaling

In [ ]:
nominal_cols = ['Marital status', 'Application mode', 'Course',
                'Nacionality', "Mother's occupation", "Father's occupation"]

binary_cols = ['Displaced', 'Educational special needs', 'Debtor',
               'Tuition fees up to date', 'Gender', 'Scholarship holder',
               'International', 'Daytime/evening attendance', 'zero_approval_flag']

standard_cols = ['Admission grade', 'Previous qualification (grade)']

robust_cols = ['Curricular units 1st sem (enrolled)', 'Curricular units 2nd sem (enrolled)',
               'Curricular units 1st sem (evaluations)', 'Curricular units 2nd sem (evaluations)',
               'total_enrolled', 'total_approved']

minmax_cols = ['Unemployment rate', 'Inflation rate', 'GDP']

engineered_numeric = ['approval_rate_s1', 'approval_rate_s2', 'grade_trend', 'parent_edu_max']

print('Nominal:', len(nominal_cols), '| Binary:', len(binary_cols),
      '| Standard-scaled:', len(standard_cols), '| Robust-scaled:', len(robust_cols),
      '| MinMax-scaled:', len(minmax_cols), '| Engineered numeric:', len(engineered_numeric))

Quick preview of one-hot encoding size before it's folded into the full pipeline:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe_preview = OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False)
encoded_preview = ohe_preview.fit_transform(df[nominal_cols])
print('Encoded shape:', encoded_preview.shape)

### 3.5 Correlation Filtering (drop redundant numeric columns)

In [ ]:
corr = df.select_dtypes('number').corr().abs()
high_corr_pairs = [(c1, c2) for c1 in corr.columns for c2 in corr.columns
                   if c1 < c2 and corr.loc[c1, c2] > 0.9]
print('Highly correlated pairs (|r| > 0.9):', len(high_corr_pairs))
for pair in high_corr_pairs:
    print(' ', pair[0], '<->', pair[1])

In [ ]:
# Drop one column from each highly correlated pair (keep the more directly interpretable one)
cols_to_drop = ['Curricular units 1st sem (evaluations)', 'Curricular units 2nd sem (evaluations)']
df = df.drop(columns=cols_to_drop)
robust_cols = [c for c in robust_cols if c not in cols_to_drop]

## 4. Assemble the Full Pipeline

Fitting happens only on the training split; the same fitted transformers are then applied to the
test split with `.transform()`, which prevents test-set leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

X = df.drop(columns=['Target', 'age_bucket'])  # age_bucket kept for reference/EDA only here
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
])

preprocessor = ColumnTransformer([
    ('robust', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', RobustScaler())]), robust_cols),
    ('standard', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), standard_cols),
    ('minmax', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', MinMaxScaler())]), minmax_cols),
    ('engineered', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), engineered_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01), nominal_cols),
    ('bin', SimpleImputer(strategy='most_frequent'), binary_cols),
], remainder='drop')

pipe = Pipeline([('preprocessing', preprocessor)])

X_train_ready = pipe.fit_transform(X_train)
X_test_ready = pipe.transform(X_test)

print('Train shape:', X_train_ready.shape)
print('Test shape :', X_test_ready.shape)

## 5. (Optional) Quick Baseline Model

Not part of the preprocessing strategy itself, but useful to sanity-check that the pipeline
output is actually usable by a model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
clf.fit(X_train_ready, y_train)
preds = clf.predict(X_test_ready)

print(classification_report(y_test, preds))

## Next steps
- Save the fitted `pipe` with `joblib.dump(pipe, 'preprocessor.joblib')` so it can be reused for inference.
- Feed `X_train_ready` / `X_test_ready` into the model-comparison stage (logistic regression, gradient boosting, etc.) from the Week 1 plan.
- Log the correlation-filter and rare-category thresholds used here in the preprocessing changelog described in the strategy document.